In [5]:
import tkinter as tk
from tkinter import ttk, messagebox
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")


# =========================
# ⭐ 配置区（只改这里）
# =========================
WINDOW_TITLE = "Methane Production Prediction Software 1"

INPUT_NAMES = [
    "C (%)", "H (%)", "O (%)", "N (%)", "S (%)",
    "Ash (%)", "Solid Content (%)",
    "HTT-T (°C)", "HTT-RT (min)",
    "COD (g/L)", "TOC (g/L)",
    "AD-T (°C)", "AD-Time (day)"
]

OUTPUT_NAMES = [
    "Methane production (ml/g COD)",
    "Methane production rate (ml/(g COD·d))"
]

EXCEL_PATH = "Model 1.xlsx"


# =========================
# 1) 读取数据：训练模型 + 计算输入范围
# =========================
def load_data_and_ranges(excel_path: str):
    data = pd.read_excel(excel_path)

    # 输入：0~12列；输出：13~14列（按你现有代码约定）
    X = data.iloc[:, 0:13].to_numpy(dtype=float)
    Y = data.iloc[:, 13:15].to_numpy(dtype=float)

    # 每列 min/max（用于UI显示与校验）
    mins = np.nanmin(X, axis=0)
    maxs = np.nanmax(X, axis=0)

    ranges = []
    for i in range(X.shape[1]):
        ranges.append((float(mins[i]), float(maxs[i])))

    return X, Y, ranges


def train_model(X, Y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=137,
        max_depth=63,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model

def fmt_range(vmin: float, vmax: float) -> str:
    def _fmt(x):
        # 数学意义上的整数 → 整数显示
        if float(x).is_integer():
            return str(int(x))
        # 否则保留两位小数
        return f"{x:.2f}"

    return f"[{_fmt(vmin)} , {_fmt(vmax)}]"



# =========================
# 2) GUI
# =========================
class PredictorGUI:
    def __init__(self, root, model, ranges):
        self.root = root
        self.model = model
        self.ranges = ranges  # list of (min,max) length=13

        self.root.title(WINDOW_TITLE)
        self.root.geometry("980x620")
        self.root.minsize(920, 580)

        # ---------- Style 美化 ----------
        style = ttk.Style()
        # 尽量用系统可用主题（不同电脑不一样）
        try:
            style = ttk.Style()
            #style.theme_use("clam")
        except:
            pass

        default_font = ("Segoe UI", 10)  # Windows上更像软件；若你想改成 Times New Roman 也可以
        self.root.option_add("*Font", default_font)

        style.configure("Title.TLabel", font=("Segoe UI", 13, "bold"))
        style.configure("Group.TLabelframe", padding=10)
        style.configure("Group.TLabelframe.Label", font=("Segoe UI", 10, "bold"))
        style.configure("Hint.TLabel", foreground="#666666")
        style.configure("Out.TEntry", fieldbackground="#F3F5F7")
        style.configure("Primary.TButton", font=("Segoe UI", 10, "bold"))

        # ---------- 顶部标题 ----------
        header = ttk.Frame(root, padding=(14, 10, 14, 6))
        header.pack(fill="x")
        ttk.Label(header, text=WINDOW_TITLE, style="Title.TLabel").pack(side="left")
        ttk.Label(
            header,
            text="Model: Random Forest | Inputs: 13 | Outputs: 2",
            style="Hint.TLabel"
        ).pack(side="right")

        # ---------- 主容器 ----------
        container = ttk.Frame(root, padding=(14, 8, 14, 14))
        container.pack(fill="both", expand=True)

        # 上部：输入
        input_frame = ttk.LabelFrame(container, text="Inputs", style="Group.TLabelframe")
        input_frame.pack(fill="both", expand=True)

        # 下部：输出
        output_frame = ttk.LabelFrame(container, text="Outputs", style="Group.TLabelframe")
        output_frame.pack(fill="x", pady=(10, 0))

        # 输入两列
        left_col = ttk.Frame(input_frame)
        right_col = ttk.Frame(input_frame)
        left_col.grid(row=0, column=0, sticky="nsew", padx=(0, 14))
        right_col.grid(row=0, column=1, sticky="nsew")

        input_frame.columnconfigure(0, weight=1)
        input_frame.columnconfigure(1, weight=1)
        input_frame.rowconfigure(0, weight=1)

        # 四个分组
        g1 = ttk.LabelFrame(left_col, text="Biomass characteristics", style="Group.TLabelframe")
        g1.pack(fill="both", expand=True, pady=(0, 10))

        g2 = ttk.LabelFrame(right_col, text="HTT conditions", style="Group.TLabelframe")
        g3 = ttk.LabelFrame(right_col, text="AP properties", style="Group.TLabelframe")
        g4 = ttk.LabelFrame(right_col, text="AD parameters", style="Group.TLabelframe")
        g2.pack(fill="x", pady=(0, 10))
        g3.pack(fill="x", pady=(0, 10))
        g4.pack(fill="x")

        # 输入框列表
        self.entries = [None] * 13
        self.range_labels = [None] * 13

        def add_row(parent, idx, label_text, r):
            # 三列：名称 | 输入框 | 范围
            lab = ttk.Label(parent, text=label_text)
            lab.grid(row=r, column=0, sticky="w", pady=5, padx=(0, 8))

            ent = ttk.Entry(parent, width=18)  # ✅ 缩小输入框
            ent.grid(row=r, column=1, sticky="w", pady=5)

            vmin, vmax = self.ranges[idx]
            rlab = ttk.Label(parent, text=fmt_range(vmin, vmax), style="Hint.TLabel")
            rlab.grid(row=r, column=2, sticky="w", pady=5, padx=(10, 0))

            parent.columnconfigure(0, weight=0)
            parent.columnconfigure(1, weight=0)
            parent.columnconfigure(2, weight=1)

            self.entries[idx] = ent
            self.range_labels[idx] = rlab

        # 组1：0-6
        for r, i in enumerate(range(0, 7)):
            add_row(g1, i, INPUT_NAMES[i], r)

        # 组2：7-8
        for r, i in enumerate(range(7, 9)):
            add_row(g2, i, INPUT_NAMES[i], r)

        # 组3：9-10
        for r, i in enumerate(range(9, 11)):
            add_row(g3, i, INPUT_NAMES[i], r)

        # 组4：11-12
        for r, i in enumerate(range(11, 13)):
            add_row(g4, i, INPUT_NAMES[i], r)

        # ---------- 输出区 ----------
        self.out1_var = tk.StringVar()
        self.out2_var = tk.StringVar()

        ttk.Label(output_frame, text=OUTPUT_NAMES[0]).grid(row=0, column=0, sticky="w", pady=6, padx=(0, 10))
        out1 = ttk.Entry(output_frame, textvariable=self.out1_var, state="readonly", width=36, style="Out.TEntry")
        out1.grid(row=0, column=1, sticky="w", pady=6)

        ttk.Label(output_frame, text=OUTPUT_NAMES[1]).grid(row=1, column=0, sticky="w", pady=6, padx=(0, 10))
        out2 = ttk.Entry(output_frame, textvariable=self.out2_var, state="readonly", width=36, style="Out.TEntry")
        out2.grid(row=1, column=1, sticky="w", pady=6)

        output_frame.columnconfigure(0, weight=0)
        output_frame.columnconfigure(1, weight=1)

        # 分割线
        sep = ttk.Separator(output_frame, orient="horizontal")
        sep.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(10, 8))

        # 按钮区
        btn_frame = ttk.Frame(output_frame)
        btn_frame.grid(row=3, column=0, columnspan=2, sticky="w")

        ttk.Button(btn_frame, text="Predict", style="Primary.TButton", command=self.predict).grid(row=0, column=0, padx=(0, 10))
        ttk.Button(btn_frame, text="Clear", command=self.clear).grid(row=0, column=1)

        # 提示
        self.hint = tk.StringVar(value="Please input all features within the suggested ranges, then click Predict.")
        ttk.Label(output_frame, textvariable=self.hint, style="Hint.TLabel") \
            .grid(row=4, column=0, columnspan=2, sticky="w", pady=(10, 0))

    def _read_inputs(self):
        values = []
        out_of_range = []

        for i, (name, ent) in enumerate(zip(INPUT_NAMES, self.entries)):
            txt = ent.get().strip()
            if txt == "":
                raise ValueError(f"{name} is empty.")
            try:
                v = float(txt)
            except:
                raise ValueError(f"{name} must be numeric.")

            vmin, vmax = self.ranges[i]
            # 允许轻微超出？这里严格提示，但不强制拦截（你也可改成强制 raise）
            if v < vmin or v > vmax:
                out_of_range.append(f"{name}: {v} not in {fmt_range(vmin, vmax)}")

            values.append(v)

        X = np.array(values, dtype=float).reshape(1, -1)

        # 如果有超范围，弹窗提醒，但仍允许继续预测（避免用户必须改到范围内才能试）
        if out_of_range:
            messagebox.showwarning(
                "Input Range Warning",
                "Some inputs are outside the ranges observed in the training data:\n\n"
                + "\n".join(out_of_range)
                + "\n\nPrediction may be less reliable."
            )

        return X

    def predict(self):
        try:
            X = self._read_inputs()
            y = self.model.predict(X)
            self.out1_var.set(f"{y[0, 0]:.6f}")
            self.out2_var.set(f"{y[0, 1]:.6f}")
            self.hint.set("Prediction completed.")
        except Exception as e:
            messagebox.showerror("Prediction Error", str(e))
            self.hint.set("Prediction failed. Please check inputs.")

    def clear(self):
        for ent in self.entries:
            ent.delete(0, tk.END)
        self.out1_var.set("")
        self.out2_var.set("")
        self.hint.set("Cleared.")


# =========================
# 3) main
# =========================
def main():
    # 读取数据 + 范围
    X, Y, ranges = load_data_and_ranges(EXCEL_PATH)

    # 训练模型
    model = train_model(X, Y)

    # GUI
    root = tk.Tk()
    PredictorGUI(root, model, ranges)
    root.mainloop()


if __name__ == "__main__":
    main()
